In [ ]:
!pip -q install transformers accelerate qwen-vl-utils pillow pandas tqdm scikit-learn

In [ ]:
import os
import pandas as pd
from PIL import Image
from tqdm import tqdm
from sklearn.metrics import classification_report

from google.colab import drive
drive.mount("/content/drive")

!pip -q install transformers accelerate bitsandbytes pillow pandas tqdm scikit-learn

import os
import pandas as pd
from PIL import Image
from tqdm import tqdm
from sklearn.metrics import classification_report

from google.colab import drive
drive.mount("/content/drive")

SOFT_DIR = "/content/drive/MyDrive/247C Project/data/soft_story_images/images"
NON_DIR  = "/content/drive/MyDrive/247C Project/data/non_soft_story_images/images"

def list_images(folder):
    exts = (".jpg", ".jpeg", ".png", ".webp")
    return sorted([
        os.path.join(folder, f)
        for f in os.listdir(folder)
        if f.lower().endswith(exts)
    ])

# Only images directly inside each folder
soft_paths = list_images(SOFT_DIR)
non_paths = list_images(NON_DIR)

print("Soft images:", len(soft_paths))
print("Non images:", len(non_paths))
print("Example soft path:", soft_paths[0] if soft_paths else "None found")
print("Example non path:", non_paths[0] if non_paths else "None found")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Soft images: 2225
Non images: 2192
Example soft path: /content/drive/MyDrive/247C Project/data/soft_story_images/images/0_view1.jpg
Example non path: /content/drive/MyDrive/247C Project/data/non_soft_story_images/images/1000_view0.jpg


In [ ]:
import torch
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info

MODEL_ID = "Qwen/Qwen2-VL-2B-Instruct"

model = Qwen2VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto"
)

processor = AutoProcessor.from_pretrained(MODEL_ID)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

...

The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
# the prompt I used for the results
PROMPT = """
You are classifying a street-view image of one building.

Task:
Decide whether the building visually appears to be a SOFT_STORY building or a NON_SOFT_STORY building.

SOFT_STORY means:
- the ground floor appears noticeably more open, weaker, or less laterally stiff than the upper floors
- examples include tuck-under parking, shops, large garage openings, or very limited solid wall area on the first floor

NON_SOFT_STORY means:
- the building does not clearly show that pattern

Use only visible architectural and structural cues.
Do not use guessed information about age, number of apartments, retrofit completion, or ordinance coverage.

NO CLEAR CLASSIFICATION means:
if the image is not a building, heavily occluded, and not possible to classify as soft story or not.

Output format:
Return exactly one of these two strings:
SOFT_STORY
NON_SOFT_STORY
NO CLEAR CLASSIFICATION
"""

In [ ]:
# # Prompt with 3 classifications (for Varun)
# PROMPT = """
# You are classifying a street-view image of one building.

# Task:
# Decide whether the main building in the image visually appears to be a SOFT_STORY building, a NON_SOFT_STORY building, or NO_CLEAR_CLASSIFICATION.

# Definitions:

# SOFT_STORY:
# - the ground floor appears noticeably more open, weaker, or less laterally stiff than the upper floors
# - examples include tuck-under parking, shops, large garage openings, or very limited solid wall area on the first floor

# NON_SOFT_STORY:
# - the building does not clearly show that pattern
# - the first story appears similarly enclosed or similarly stiff compared with the upper floors

# NO_CLEAR_CLASSIFICATION:
# - the image does not clearly show a building
# - the building is heavily occluded
# - the first story is not visible enough to assess
# - the building is too far away or too blurry
# - multiple buildings are visible and the main building is unclear
# - the visible evidence is insufficient to confidently classify as SOFT_STORY or NON_SOFT_STORY

# Rules:
# - Use only visible architectural and structural cues.
# - Do not infer building age, number of apartments, retrofit completion, or ordinance eligibility.
# - Focus on the relative openness or weakness of the first story compared with the upper stories.

# Output format:
# Return exactly one of these three strings and nothing else:
# SOFT_STORY
# NON_SOFT_STORY
# NO_CLEAR_CLASSIFICATION
# """

In [ ]:
def classify_qwen(image_path):
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image_path},
                {"type": "text", "text": PROMPT},
            ],
        }
    ]

    text = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    image_inputs, video_inputs = process_vision_info(messages)

    inputs = processor(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt"
    ).to("cuda")

    with torch.no_grad():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=10
        )

    generated_ids_trimmed = [
        out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
    ]

    output_text = processor.batch_decode(
        generated_ids_trimmed,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False
    )[0].strip().upper()

    if "SOFT_STORY" in output_text and "NON_SOFT_STORY" not in output_text:
        label = "SOFT_STORY"
    elif "NON_SOFT_STORY" in output_text:
        label = "NON_SOFT_STORY"
    else:
        label = "NON_SOFT_STORY"  # conservative fallback

    return label, output_text

In [ ]:
rows = []

for p in tqdm(soft_paths, desc="soft"):
    pred, raw = classify_qwen(p)
    rows.append({
        "path": p,
        "folder": "soft_story",
        "true_label": "SOFT_STORY",
        "pred_label": pred,
        "raw_output": raw
    })

for p in tqdm(non_paths, desc="non"):
    pred, raw = classify_qwen(p)
    rows.append({
        "path": p,
        "folder": "non_soft_story",
        "true_label": "NON_SOFT_STORY",
        "pred_label": pred,
        "raw_output": raw
    })

df = pd.DataFrame(rows)
df.to_csv("qwen2vl_results.csv", index=False)

print("OVERALL")
print(classification_report(df["true_label"], df["pred_label"], digits=3))

soft_df = df[df["folder"] == "soft_story"]
non_df = df[df["folder"] == "non_soft_story"]

print("SOFT folder accuracy:", (soft_df["true_label"] == soft_df["pred_label"]).mean())
print("NON  folder accuracy:", (non_df["true_label"] == non_df["pred_label"]).mean())

df.head()


non: 100%|██████████| 2192/2192 [38:00<00:00,  1.04s/it]
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(

OVERALL
                precision    recall  f1-score   support

NON_SOFT_STORY      0.000     0.000     0.000      2192
    SOFT_STORY      0.504     1.000     0.670      2225

      accuracy                          0.504      4417
     macro avg      0.252     0.500     0.335      4417
  weighted avg      0.254     0.504     0.337      4417

SOFT folder accuracy: 1.0
NON  folder accuracy: 0.0


,path,folder,true_label,pred_label,raw_output
0,/content/drive/MyDrive/247C Project/data/soft_...,soft_story,SOFT_STORY,SOFT_STORY,SOFT_STORY
1,/content/drive/MyDrive/247C Project/data/soft_...,soft_story,SOFT_STORY,SOFT_STORY,SOFT_STORY
2,/content/drive/MyDrive/247C Project/data/soft_...,soft_story,SOFT_STORY,SOFT_STORY,SOFT_STORY
3,/content/drive/MyDrive/247C Project/data/soft_...,soft_story,SOFT_STORY,SOFT_STORY,SOFT_STORY
4,/content/drive/MyDrive/247C Project/data/soft_...,soft_story,SOFT_STORY,SOFT_STORY,SOFT_STORY


In [ ]:
from sklearn.metrics import classification_report, accuracy_score

print("OVERALL METRICS\n")

print(classification_report(
    df["true_label"],
    df["pred_label"],
    digits=3
))

print("Accuracy:", accuracy_score(df["true_label"], df["pred_label"]))

OVERALL METRICS

                precision    recall  f1-score   support

NON_SOFT_STORY      0.000     0.000     0.000      2192
    SOFT_STORY      0.504     1.000     0.670      2225

      accuracy                          0.504      4417
     macro avg      0.252     0.500     0.335      4417
  weighted avg      0.254     0.504     0.337      4417

Accuracy: 0.5037355671270093


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
from sklearn.metrics import confusion_matrix

labels = ["SOFT_STORY", "NON_SOFT_STORY"]

cm = confusion_matrix(
    df["true_label"],
    df["pred_label"],
    labels=labels
)

print("Confusion Matrix:")
print(cm)

Confusion Matrix:
[[2225    0]
 [2192    0]]
